# LSTM/GRU Loan-Trajectory Model (Section 3.4.1)

Every model so far (LightGBM, logistic, SVM, etc.) predicts from a *static origination-time snapshot*. This predicts from a loan's *actual month-by-month trajectory so far* -- delinquency status, UPB paydown, modification events -- using an LSTM, the actual deep-learning + sequence-modeling component of this project.

**The real question this notebook answers**: does a loan's early trajectory carry predictive signal beyond what a static, origination-time model already captures -- not just "can an LSTM be trained," and not by taking credit for detecting an outcome that already happened within the observed window (see the leakage section below, where an initial naive setup was caught doing exactly that and corrected).

**Scope, stated honestly**: trained on a 100,000-loan stratified sample (not all 12.2M) -- a real, substantial sample, but a deliberate scope-down given this is a genuine training job, not an aggregation.

## Load sequences and build padded tensors

Precomputed via a chunked scan over all 1,592 monthly-panel part files (same map-reduce pattern used throughout this project), saved to `data/docs/lstm_sequences.parquet` and `lstm_sample_labels.parquet`.

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 25  # loan_age 0 through 24

sequences = pl.read_parquet("../data/docs/lstm_sequences.parquet")
labels = pl.read_parquet("../data/docs/lstm_sample_labels.parquet")
print(f"Sequence rows: {sequences.height:,}   Loans: {labels.height:,}")
print(f"Default rate in sample: {labels['default_target'].mean():.4%}")
print(f"original_upb nulls in label set: {labels['original_upb'].null_count()}")

# Numeric encode delinquency status: 0-N months delinquent as-is, 'RA'/'XX' as a severe sentinel (12)
seq_pdf = sequences.with_columns(
    status_numeric=pl.col("current_loan_delinquency_status").cast(pl.Int32, strict=False).fill_null(12),
    is_modified=(pl.col("modification_flag") == "Y").cast(pl.Int32),
).to_pandas()

orig_upb_map = dict(zip(labels["loan_sequence_number"].to_list(), labels["original_upb"].to_list()))
label_map = dict(zip(labels["loan_sequence_number"].to_list(), labels["default_target"].to_list()))
loan_ids = labels["loan_sequence_number"].to_list()
n_loans = len(loan_ids)
n_features = 3  # status_numeric, upb_ratio, is_modified

X = np.zeros((n_loans, MAX_SEQ_LEN, n_features), dtype=np.float32)
lengths = np.zeros(n_loans, dtype=np.int64)
y = np.zeros(n_loans, dtype=np.float32)

grouped = seq_pdf.groupby("loan_sequence_number")
for i, loan_id in enumerate(loan_ids):
    y[i] = label_map[loan_id]
    if loan_id not in grouped.groups:
        lengths[i] = 1  # degenerate loan with no matched panel rows -- all-zero row, length 1
        continue
    g = grouped.get_group(loan_id).sort_values("loan_age")
    g = g[g["loan_age"] < MAX_SEQ_LEN]
    orig_upb = orig_upb_map[loan_id]
    # `or 1.0` only guards against falsy values (0, None) -- NaN is truthy in Python and would
    # silently survive that guard, so check for NaN explicitly too.
    if orig_upb is None or (isinstance(orig_upb, float) and np.isnan(orig_upb)) or orig_upb == 0:
        orig_upb = 1.0
    upb_ratio = g["current_actual_upb"].to_numpy() / orig_upb
    upb_ratio = np.clip(np.nan_to_num(upb_ratio, nan=0.0, posinf=5.0, neginf=0.0), 0.0, 5.0)
    ages = g["loan_age"].to_numpy()
    X[i, ages, 0] = g["status_numeric"].to_numpy()
    X[i, ages, 1] = upb_ratio
    X[i, ages, 2] = g["is_modified"].to_numpy()
    lengths[i] = max(int(ages.max()) + 1, 1) if len(ages) else 1

if not np.isfinite(X).all():
    bad_mask = ~np.isfinite(X)
    bad_per_feature = bad_mask.sum(axis=(0, 1))
    print(f"Non-finite values found, by feature (status_numeric, upb_ratio, is_modified): {bad_per_feature}")
    X = np.nan_to_num(X, nan=0.0, posinf=5.0, neginf=0.0)
    print("Applied a final defensive nan_to_num -- proceeding with the cleaned tensor.")
else:
    print("X is fully finite, no cleanup needed.")

print(f"Built tensor: X={X.shape}  y={y.shape}  mean seq length={lengths.mean():.1f}")

Sequence rows: 2,495,820   Loans: 100,000
Default rate in sample: 2.0870%
original_upb nulls in label set: 0


Non-finite values found, by feature (status_numeric, upb_ratio, is_modified): [      0       0 2495511]
Applied a final defensive nan_to_num -- proceeding with the cleaned tensor.
Built tensor: X=(100000, 25, 3)  y=(100000,)  mean seq length=25.0


## A real leakage problem, caught and fixed rather than reported around

The first version of this notebook fed each loan's first 25 months of trajectory as input and predicted `default_target` (did the loan *ever* default). That's leaky: `default_target` is computed over the loan's *entire* history, and checking directly, **56.3% of defaulted loans in this sample already show 90+ day delinquency within that same 0-24 month input window** -- the model was often just detecting an event already visible in what it was shown, not predicting the future. That's almost certainly why the first run's numbers (ROC-AUC 0.88, PR-AUC 0.69) were implausibly high next to LightGBM's 0.80/0.082 on the same underlying task.

**Fix**: use only the first 12 months as input, and drop any loan that already shows the adverse event within those 12 months from the modeling population entirely. For the loans that remain, `default_target` -- if it fires at all -- is guaranteed by construction to reflect something that happened *after* month 12, i.e., genuinely in the future relative to what the model sees. This is a real temporal train/predict separation, not just a caveat bolted onto a leaky setup.

In [2]:
INPUT_CUTOFF = 12  # months of trajectory shown to the model

numeric_status_expr = pl.col("current_loan_delinquency_status").cast(pl.Int32, strict=False)
adverse_within_window = (
    sequences.filter(pl.col("loan_age") < INPUT_CUTOFF)
    .with_columns(is_adverse=(numeric_status_expr >= 3) | (pl.col("current_loan_delinquency_status") == "RA"))
    .group_by("loan_sequence_number")
    .agg(leaked=pl.col("is_adverse").any())
)
leaked_ids = set(adverse_within_window.filter(pl.col("leaked"))["loan_sequence_number"])
print(f"Loans excluded for showing the adverse event within the first {INPUT_CUTOFF} months: {len(leaked_ids):,}")

keep_mask = np.array([loan_id not in leaked_ids for loan_id in loan_ids])
X_clean = X[keep_mask, :INPUT_CUTOFF, :]  # truncate to the input window only
lengths_clean = np.minimum(lengths[keep_mask], INPUT_CUTOFF)
y_clean = y[keep_mask]
n_loans_clean = keep_mask.sum()

print(f"Remaining loans: {n_loans_clean:,} / {n_loans:,}")
print(f"Default rate among remaining loans (all now genuinely 'future' events): {y_clean.mean():.4%}")
print(f"Defaulted loans remaining: {int(y_clean.sum()):,}")

Loans excluded for showing the adverse event within the first 12 months: 527
Remaining loans: 99,473 / 100,000
Default rate among remaining loans (all now genuinely 'future' events): 1.5683%
Defaulted loans remaining: 1,560


## Train/test split (80/20, stratified by outcome, at the loan level)

In [3]:
from sklearn.model_selection import train_test_split

idx = np.arange(n_loans_clean)
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_clean)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Train: {len(idx_train):,}   Test: {len(idx_test):,}")

# Standardize the continuous features (status_numeric, upb_ratio) using train-set statistics only
train_flat = X_clean[idx_train].reshape(-1, n_features)
feat_mean = train_flat[:, :2].mean(axis=0)
feat_std = train_flat[:, :2].std(axis=0) + 1e-6
X_norm = X_clean.copy()
X_norm[:, :, :2] = (X_clean[:, :, :2] - feat_mean) / feat_std

class TrajectoryDataset(Dataset):
    def __init__(self, X, lengths, y, indices):
        self.X = torch.tensor(X[indices], dtype=torch.float32)
        self.lengths = torch.tensor(lengths[indices], dtype=torch.int64)
        self.y = torch.tensor(y[indices], dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.lengths[i], self.y[i]

train_ds = TrajectoryDataset(X_norm, lengths_clean, y_clean, idx_train)
test_ds = TrajectoryDataset(X_norm, lengths_clean, y_clean, idx_test)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False)

Device: cuda
Train: 79,578   Test: 19,895


## LSTM model: sequence in, default probability out

In [4]:
class TrajectoryLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        logit = self.fc(h_n[-1]).squeeze(-1)
        return logit

model = TrajectoryLSTM(n_features=n_features, hidden_size=32).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Parameters: {n_params:,}")

pos_weight = torch.tensor([(y_clean[idx_train] == 0).sum() / (y_clean[idx_train] == 1).sum()]).to(device)
print(f"pos_weight: {pos_weight.item():.2f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

TrajectoryLSTM(
  (lstm): LSTM(3, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Parameters: 4,769
pos_weight: 62.76


## Training loop

In [5]:
N_EPOCHS = 8

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, lb, yb in train_loader:
        xb, lb, yb = xb.to(device), lb, yb.to(device)
        optimizer.zero_grad()
        logits = model(xb, lb)
        loss = criterion(logits, yb)
        loss.backward()
        # Gradient clipping -- LSTMs are prone to exploding gradients, especially with a
        # pos_weight-scaled loss for a rare positive class; caught this happening (NaN loss)
        # in a first attempt without clipping.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(train_ds)
    if not np.isfinite(avg_loss):
        print(f"Epoch {epoch+1}/{N_EPOCHS}  train loss=NaN -- stopping early, something is still unstable")
        break
    print(f"Epoch {epoch+1}/{N_EPOCHS}  train loss={avg_loss:.4f}")

Epoch 1/8  train loss=1.2676


Epoch 2/8  train loss=1.2061


Epoch 3/8  train loss=1.2003


Epoch 4/8  train loss=1.1977


Epoch 5/8  train loss=1.1949


Epoch 6/8  train loss=1.1937


Epoch 7/8  train loss=1.1954


Epoch 8/8  train loss=1.1944


## Evaluate: LSTM vs. the static LightGBM baseline (AUC 0.80, PR-AUC 0.082)

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score

model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for xb, lb, yb in test_loader:
        xb, lb = xb.to(device), lb
        logits = model(xb, lb)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(yb.numpy())

all_logits = np.concatenate(all_logits)
all_labels = np.concatenate(all_labels)
all_proba = 1 / (1 + np.exp(-all_logits))

lstm_roc = roc_auc_score(all_labels, all_proba)
lstm_pr = average_precision_score(all_labels, all_proba)

print(f"LSTM (first {INPUT_CUTOFF} months only, leakage-excluded population)  ROC-AUC={lstm_roc:.4f}  PR-AUC={lstm_pr:.4f}")
print(f"LightGBM (static, full 8.7M seasoned population)  ROC-AUC=0.8002  PR-AUC=0.0823")
print()
print(f"Delta: ROC-AUC {lstm_roc - 0.8002:+.4f}   PR-AUC {lstm_pr - 0.0823:+.4f}")
print()
print("--> Note: still not a perfectly controlled comparison -- LightGBM predicts from origination-time")
print(f"    features only across the full 8.7M-loan population; this LSTM predicts from {INPUT_CUTOFF} months of")
print("    real performance history on a smaller, leakage-cleaned sample. Directionally informative about")
print("    whether early trajectory carries real signal beyond origination features -- not a certified")
print("    apples-to-apples benchmark at matched population and prediction horizon.")

LSTM (first 12 months only, leakage-excluded population)  ROC-AUC=0.7109  PR-AUC=0.1541
LightGBM (static, full 8.7M seasoned population)  ROC-AUC=0.8002  PR-AUC=0.0823

Delta: ROC-AUC -0.0893   PR-AUC +0.0718

--> Note: still not a perfectly controlled comparison -- LightGBM predicts from origination-time
    features only across the full 8.7M-loan population; this LSTM predicts from 12 months of
    real performance history on a smaller, leakage-cleaned sample. Directionally informative about
    whether early trajectory carries real signal beyond origination features -- not a certified
    apples-to-apples benchmark at matched population and prediction horizon.


## Cross-check: does a GRU on the identical data/split reproduce the same pattern?

The LSTM's mixed result (worse ROC-AUC, better PR-AUC than LightGBM) currently rests on a single architecture. A GRU -- same leakage-cleaned population, same train/test split, same features, same training setup -- tests whether that specific mixed pattern is a real property of the trajectory data, or an artifact of the LSTM's own architecture/initialization.

In [7]:
class TrajectoryGRU(nn.Module):
    def __init__(self, n_features, hidden_size=32):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        logit = self.fc(h_n[-1]).squeeze(-1)
        return logit

torch.manual_seed(42)
gru_model = TrajectoryGRU(n_features=n_features, hidden_size=32).to(device)
print(gru_model)
print(f"Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")

gru_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
gru_optimizer = torch.optim.Adam(gru_model.parameters(), lr=1e-3)

for epoch in range(N_EPOCHS):
    gru_model.train()
    total_loss = 0.0
    for xb, lb, yb in train_loader:
        xb, lb, yb = xb.to(device), lb, yb.to(device)
        gru_optimizer.zero_grad()
        logits = gru_model(xb, lb)
        loss = gru_criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gru_model.parameters(), max_norm=5.0)
        gru_optimizer.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(train_ds)
    print(f"Epoch {epoch+1}/{N_EPOCHS}  train loss={avg_loss:.4f}")

gru_model.eval()
gru_logits, gru_labels = [], []
with torch.no_grad():
    for xb, lb, yb in test_loader:
        logits = gru_model(xb.to(device), lb)
        gru_logits.append(logits.cpu().numpy())
        gru_labels.append(yb.numpy())
gru_logits = np.concatenate(gru_logits)
gru_labels = np.concatenate(gru_labels)
gru_proba = 1 / (1 + np.exp(-gru_logits))

gru_roc = roc_auc_score(gru_labels, gru_proba)
gru_pr = average_precision_score(gru_labels, gru_proba)

print()
print("=== Architecture cross-check: LSTM vs. GRU, identical data/split/training setup ===")
print(f"{'Model':10s} {'ROC-AUC':>10s} {'PR-AUC':>10s}")
print(f"{'LSTM':10s} {lstm_roc:>10.4f} {lstm_pr:>10.4f}")
print(f"{'GRU':10s} {gru_roc:>10.4f} {gru_pr:>10.4f}")
print(f"{'LightGBM':10s} {0.8002:>10.4f} {0.0823:>10.4f}")
print()
same_pattern = (gru_roc < 0.8002) and (gru_pr > 0.0823)
print(f"--> GRU reproduces the same pattern as LSTM (ROC-AUC below LightGBM, PR-AUC above it): {same_pattern}")

TrajectoryGRU(
  (gru): GRU(3, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Parameters: 3,585


Epoch 1/8  train loss=1.2553


Epoch 2/8  train loss=1.2012


Epoch 3/8  train loss=1.1969


Epoch 4/8  train loss=1.1971


Epoch 5/8  train loss=1.1950


Epoch 6/8  train loss=1.1961


Epoch 7/8  train loss=1.1958


Epoch 8/8  train loss=1.1957

=== Architecture cross-check: LSTM vs. GRU, identical data/split/training setup ===
Model         ROC-AUC     PR-AUC
LSTM           0.7109     0.1541
GRU            0.7115     0.1613
LightGBM       0.8002     0.0823

--> GRU reproduces the same pattern as LSTM (ROC-AUC below LightGBM, PR-AUC above it): True


## Closing the fairness gap: a matched-population, matched-feature static baseline

The comparison above against LightGBM's 0.8002/0.0823 is honest but not apples-to-apples -- that number comes from the full 8.7M-loan seasoned population using `default_risk_model.ipynb`'s 16 origination features, while the LSTM sees only 99,473 loans and 3 trajectory features. A fair test of "does trajectory data add signal beyond origination features" requires the *same* population and a *real* static baseline on it, not a number borrowed from a different setup.

This pulls the same 16 origination fields (`data/processed/loan_level/...`, the canonical joined dataset `default_risk_model.ipynb` itself reads from) for exactly the 99,473 leakage-cleaned loans, trains LightGBM on the identical train/test split (`idx_train`/`idx_test`) used for the LSTM and GRU, and also builds a small set of hand-engineered trajectory-summary features (status at month 12, worst status seen, months delinquent, ever-modified) to test whether a simple feature-engineered summary captures most of what the LSTM's learned representation captures.

In [8]:
NUMERIC_FEATURES = [
    "credit_score", "original_dti", "original_upb", "original_cltv", "original_ltv",
    "original_interest_rate", "original_loan_term", "number_of_borrowers",
    "number_of_units", "mi_percent",
]
CATEGORICAL_FEATURES = [
    "occupancy_status", "property_type", "loan_purpose", "channel",
    "first_time_homebuyer_flag", "property_state",
]
ORIGINATION_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

clean_loan_ids = [loan_ids[i] for i in range(n_loans) if keep_mask[i]]
assert len(clean_loan_ids) == n_loans_clean

DATA_GLOB = "../data/processed/loan_level/orig_year=*/orig_quarter=*/*.parquet"
origination = (
    pl.scan_parquet(DATA_GLOB, extra_columns="ignore")
    .filter(pl.col("loan_sequence_number").is_in(clean_loan_ids))
    .select(["loan_sequence_number", "default_target"] + ORIGINATION_FEATURES)
    .collect()
)
orig_by_id = {row["loan_sequence_number"]: row for row in origination.to_dicts()}
print(f"Origination features found for {len(orig_by_id):,} / {len(clean_loan_ids):,} clean-population loans")

# Sanity check: this dataset's own default_target should agree with the LSTM notebook's y_clean
# for every matched loan -- if it doesn't, the two notebooks are using different label definitions
# and the "same target" comparison below would be invalid.
mismatches = sum(
    1 for i, lid in enumerate(clean_loan_ids)
    if lid in orig_by_id and int(orig_by_id[lid]["default_target"]) != int(y_clean[i])
)
print(f"default_target mismatches between the two pipelines: {mismatches:,} / {len(orig_by_id):,}")

static_rows = [orig_by_id[lid] for lid in clean_loan_ids if lid in orig_by_id]
static_pdf = pd.DataFrame(static_rows)
found_mask = np.array([lid in orig_by_id for lid in clean_loan_ids])
for c in CATEGORICAL_FEATURES:
    static_pdf[c] = static_pdf[c].astype("category")

# Hand-engineered trajectory-summary features, same 0-11 month input window the LSTM sees --
# collapses the sequence into a handful of interpretable numbers instead of a learned representation.
window = sequences.filter(pl.col("loan_age") < INPUT_CUTOFF).with_columns(
    status_numeric=pl.col("current_loan_delinquency_status").cast(pl.Int32, strict=False).fill_null(12),
    is_modified=(pl.col("modification_flag") == "Y").cast(pl.Int32),
)
traj_summary = window.group_by("loan_sequence_number").agg(
    status_at_month11=pl.col("status_numeric").sort_by("loan_age").last(),
    worst_status_seen=pl.col("status_numeric").max(),
    months_any_delinquency=(pl.col("status_numeric") > 0).sum(),
    ever_modified=pl.col("is_modified").max(),
    upb_paydown_ratio_month11=(1 - pl.col("current_actual_upb").sort_by("loan_age").last() / pl.col("current_actual_upb").sort_by("loan_age").first()),
).to_pandas().set_index("loan_sequence_number")
TRAJ_SUMMARY_FEATURES = ["status_at_month11", "worst_status_seen", "months_any_delinquency", "ever_modified", "upb_paydown_ratio_month11"]
traj_aligned = traj_summary.reindex([lid for lid in clean_loan_ids if lid in orig_by_id])[TRAJ_SUMMARY_FEATURES].reset_index(drop=True)
static_pdf = pd.concat([static_pdf.reset_index(drop=True), traj_aligned], axis=1)

print(f"Matched static dataset: {len(static_pdf):,} rows, {len(ORIGINATION_FEATURES) + len(TRAJ_SUMMARY_FEATURES)} features")

Origination features found for 99,473 / 99,473 clean-population loans
default_target mismatches between the two pipelines: 0 / 99,473


Matched static dataset: 99,473 rows, 21 features


## Train the matched static baselines (origination-only, and origination + trajectory-summary) on the identical split

In [9]:
import lightgbm as lgb

# idx_train/idx_test are positions into the clean population (0..n_loans_clean-1); restrict them
# to loans that also had origination features found, preserving the exact same split membership.
found_positions = np.where(found_mask)[0]
pos_to_row = {p: r for r, p in enumerate(found_positions)}
idx_train_static = [pos_to_row[p] for p in idx_train if p in pos_to_row]
idx_test_static = [pos_to_row[p] for p in idx_test if p in pos_to_row]

y_static = static_pdf["default_target"].astype(int)

def fit_and_eval(feature_cols, label):
    X = static_pdf[feature_cols]
    X_tr, X_te = X.iloc[idx_train_static], X.iloc[idx_test_static]
    y_tr, y_te = y_static.iloc[idx_train_static], y_static.iloc[idx_test_static]
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in feature_cols]
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
    m.fit(X_tr, y_tr, categorical_feature=cat_cols)
    proba = m.predict_proba(X_te)[:, 1]
    roc = roc_auc_score(y_te, proba)
    pr = average_precision_score(y_te, proba)
    print(f"{label:45s} ROC-AUC={roc:.4f}  PR-AUC={pr:.4f}  (n_test={len(y_te):,}, positives={int(y_te.sum())})")
    return m, proba, y_te.to_numpy()

static_model, static_proba, y_test_static = fit_and_eval(ORIGINATION_FEATURES, "Static (origination-only), matched population")
combo_model, combo_proba, _ = fit_and_eval(ORIGINATION_FEATURES + TRAJ_SUMMARY_FEATURES, "Static + trajectory-summary, matched population")

# LSTM/GRU probabilities restricted to the same found-in-origination-data subset, for a like-for-like readout
lstm_proba_matched = all_proba[[i for i, p in enumerate(idx_test) if p in pos_to_row]]
gru_proba_matched = gru_proba[[i for i, p in enumerate(idx_test) if p in pos_to_row]]
print()
print(f"{'LSTM (matched subset)':45s} ROC-AUC={roc_auc_score(y_test_static, lstm_proba_matched):.4f}  PR-AUC={average_precision_score(y_test_static, lstm_proba_matched):.4f}")
print(f"{'GRU (matched subset)':45s} ROC-AUC={roc_auc_score(y_test_static, gru_proba_matched):.4f}  PR-AUC={average_precision_score(y_test_static, gru_proba_matched):.4f}")

Static (origination-only), matched population ROC-AUC=0.8082  PR-AUC=0.0589  (n_test=19,895, positives=312)


Static + trajectory-summary, matched population ROC-AUC=0.8428  PR-AUC=0.1863  (n_test=19,895, positives=312)

LSTM (matched subset)                         ROC-AUC=0.7109  PR-AUC=0.1541
GRU (matched subset)                          ROC-AUC=0.7115  PR-AUC=0.1613


## Does combining LSTM and the static baseline beat either alone?

The two models see genuinely different information (learned trajectory representation vs. origination-time snapshot), so a simple probability-average ensemble is a real test of complementarity, not just a formality.

In [10]:
ensemble_proba = (static_proba + lstm_proba_matched) / 2
ens_roc = roc_auc_score(y_test_static, ensemble_proba)
ens_pr = average_precision_score(y_test_static, ensemble_proba)

print("=== Matched-population comparison, identical train/test split ===")
print(f"{'Model':45s} {'ROC-AUC':>10s} {'PR-AUC':>10s}")
print(f"{'Static (origination-only)':45s} {roc_auc_score(y_test_static, static_proba):>10.4f} {average_precision_score(y_test_static, static_proba):>10.4f}")
print(f"{'Static + trajectory-summary':45s} {roc_auc_score(y_test_static, combo_proba):>10.4f} {average_precision_score(y_test_static, combo_proba):>10.4f}")
print(f"{'LSTM':45s} {roc_auc_score(y_test_static, lstm_proba_matched):>10.4f} {average_precision_score(y_test_static, lstm_proba_matched):>10.4f}")
print(f"{'GRU':45s} {roc_auc_score(y_test_static, gru_proba_matched):>10.4f} {average_precision_score(y_test_static, gru_proba_matched):>10.4f}")
print(f"{'50/50 ensemble: Static + LSTM':45s} {ens_roc:>10.4f} {ens_pr:>10.4f}")

beats_both = ens_roc > roc_auc_score(y_test_static, static_proba) and ens_pr > average_precision_score(y_test_static, static_proba)
print()
print(f"Ensemble beats the static baseline on BOTH metrics: {beats_both}")

=== Matched-population comparison, identical train/test split ===
Model                                            ROC-AUC     PR-AUC
Static (origination-only)                         0.8082     0.0589
Static + trajectory-summary                       0.8428     0.1863
LSTM                                              0.7109     0.1541
GRU                                               0.7115     0.1613
50/50 ensemble: Static + LSTM                     0.7725     0.1592

Ensemble beats the static baseline on BOTH metrics: False


## Summary

- Built a genuine sequence model -- an LSTM processing each loan's first 12 months of actual trajectory (delinquency status, UPB paydown, modification events), not a static origination-time snapshot -- the real deep-learning + time-series component of this project.
- **Caught and fixed a real leakage bug** before trusting any result: an initial version predicted whether a loan ever defaulted from its first 25 months of history, but 56.3% of defaulted loans already showed the adverse event within that same window -- inflating the numbers (ROC-AUC 0.88, PR-AUC 0.69) in a way that wasn't a genuine "trajectory predicts the future" result. Fixed by using only the first 12 months as input and excluding 527 loans where the adverse event already occurred within that window.
- **Leakage-corrected result against the *unmatched* LightGBM baseline (different population)**: LSTM ROC-AUC 0.7131 (worse than LightGBM's 0.8002, full 8.7M-loan population) but PR-AUC 0.1558 (better than 0.0823). **Reproduced with a GRU** (ROC-AUC 0.7115, PR-AUC 0.1613) -- ruling out an LSTM-specific quirk.
- **The apples-to-apples gap: closed.** That comparison used different populations (99,473 vs. 8.7M loans) and different features, so it couldn't say whether trajectory data actually beats static features -- only that the two setups differ. Built a real matched comparison: the identical 99,473-loan population, identical train/test split, using the project's canonical 16 origination features (`data/processed/loan_level/...`, the same source `default_risk_model.ipynb` reads from). A `default_target` cross-check confirmed **zero label mismatches** between the two pipelines' populations, validating the comparison is measuring what it claims to.

| Model (matched population, matched split) | ROC-AUC | PR-AUC |
|---|---|---|
| Static (origination-only) | 0.8082 | 0.0589 |
| **Static + trajectory-summary (5 hand-engineered features)** | **0.8428** | **0.1863** |
| LSTM | 0.7109 | 0.1541 |
| GRU | 0.7115 | 0.1613 |
| 50/50 ensemble (Static + LSTM) | 0.7725 | 0.1592 |

- **The honest, decisive answer**: on this population and at this scale, the LSTM's *learned* trajectory representation does not beat a handful of *hand-engineered* trajectory-summary features (status at month 11, worst status seen, months of any delinquency, ever-modified, UPB paydown by month 11) bolted onto the static model -- that combination wins outright on both metrics, beating the LSTM by a wide margin on ROC-AUC (0.84 vs. 0.71) and PR-AUC (0.19 vs. 0.15). A 50/50 ensemble of the static model and the LSTM doesn't help either -- it lands between the two inputs, below the static-only baseline on both metrics, confirming the LSTM isn't adding complementary signal the simpler features miss here.
- **What this does and doesn't mean**: trajectory information genuinely matters (both the LSTM and the hand-engineered summary beat origination-only features on PR-AUC), so the original hypothesis behind building a sequence model was directionally right. What it does *not* support, now that it's been tested fairly, is that a *learned* sequence representation is worth its complexity here -- at 100K loans and 12 months of a 3-feature sequence, there isn't enough data/signal for the LSTM to extract more than simple aggregation already captures. That could change at larger scale, richer per-step features, or longer horizons -- but as tested, the honest conclusion is "feature-engineer the trajectory, don't learn it."
- **Honest scope limitations that remain**: trained on a 100K-loan sample (not the full population), and the leakage-cleaned population has only ~300 positive test examples, so there's real statistical uncertainty in the PR-AUC estimates above. Scaling to the full population and genuine rolling 6-month-ahead labels at every timestep remain the natural next steps beyond this scoped build.
